DHANSHREE CHAUDHARI | 221A033 | BE-ADIS | 08

In [ ]:
print ("DHANSHREE CHAUDHARI|08")
!pip install gymnasium [atari]
!pip install ale.py

DHANSHREE CHAUDHARI|08
ERROR: Invalid requirement: '[atari]': Expected package name at the start of dependency specifier
    [atari]
    ^


In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
from skimage.transform import resize

In [ ]:
print("DHANSHREE CHAUDHARI | 221A033 | 08 ")

class DON(nn.Module):

    def __init__(self, input_shape, n_actions):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, 512), # Note: 3136 might need adjustment based on input_shape and conv layers
            nn.ReLU(),
            nn.Linear(512, n_actions)
        )

    def forward(self, x):
        return self.net(x / 255.0) # Normalizing by 255.0

print("DHANSHREE CHAUDHARI | 221A033 | 08")

class ReplayBuffer:

    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        # Ensure there are enough samples in the buffer
        if len(self.buffer) < batch_size:
            return None # Or raise an error, or handle appropriately
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        return (
            np.array(states),
            np.array(actions),
            np.array(rewards),
            np.array(next_states),
            np.array(dones)
        )
    def __len__(self):
        return len(self.buffer)

DHANSHREE CHAUDHARI | 221A033 | 08 
DHANSHREE CHAUDHARI | 221A033 | 08


In [ ]:
print("DHANSHREE CHAUDHARI | 221A033 | 08 ")

BATCH_SIZE = 32
GAMMA = 0.99
LR = 1e-4
BUFFER_SIZE = 100000
EPSILON_START = 1.0
EPSILON_END = 0.1 # Corrected from 6.1, assuming a typical DQN epsilon decay range
EPSILON_DECAY = 100000
TARGET_UPDATE = 1000

print("DHANSHREE CHAUDHARI | 221A033 | 08")

# Environment Setup
print("DHANSHREE CHAUDHARI | 221A033 | 08")

import ale_py # Corrected module name

env = gym.make("ALE/Pong-v5", render_mode=None) # Corrected env name and assignment
n_actions = env.action_space.n # Corrected variable assignment and attribute name

# Example input shape for Atari (stacked frames)
input_shape = (4, 84, 84) # Corrected variable assignment and standard Atari input shape


print("DHANSHREE CHAUDHARI | 221A033 | 08")

# Initialize Network
print("DHANSHREE CHAUDHARI | 221A033 | 08")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

policy_net = DON(input_shape, n_actions).to(device) # Corrected variable assignment
target_net = DON(input_shape, n_actions).to(device) # Corrected variable assignment

target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=LR) # Corrected optimizer initialization and learning rate variable
replay_buffer = ReplayBuffer(BUFFER_SIZE) # Corrected replay buffer initialization

DHANSHREE CHAUDHARI | 221A033 | 08 
DHANSHREE CHAUDHARI | 221A033 | 08
DHANSHREE CHAUDHARI | 221A033 | 08
DHANSHREE CHAUDHARI | 221A033 | 08
DHANSHREE CHAUDHARI | 221A033 | 08


Epsilon Greedy policy

In [ ]:
print("DHANSHREE CHAUDHARI | 221A033 | 08")

def epsilon_by_frame(frame_idx):
    return EPSILON_END + (EPSILON_START - EPSILON_END) * \
           np.exp(-1. * frame_idx / EPSILON_DECAY)

def select_action(state, epsilon):
    if random.random() < epsilon:
        return random.randrange(n_actions)
    else:
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        q_values = policy_net(state_tensor)
        return q_values.max(1)[1].item()


# Training Setup

print("DHANSHREE CHAUDHARI | 221A033 | 08")

def train_step():
    if len(replay_buffer) < BATCH_SIZE:
        return

    states, actions, rewards, next_states, dones = replay_buffer.sample(BATCH_SIZE)

    states_tensor = torch.tensor(states, dtype=torch.float32).to(device)
    next_states_tensor = torch.tensor(next_states, dtype=torch.float32).to(device)
    actions_tensor = torch.tensor(actions).unsqueeze(1).to(device)
    rewards_tensor = torch.tensor(rewards, dtype=torch.float32).to(device) # Explicitly cast to float32
    dones_tensor = torch.tensor(dones, dtype=torch.float32).to(device)

    q_values = policy_net(states_tensor).gather(1, actions_tensor).squeeze(1)

    next_q_values = target_net(next_states_tensor).max(1)[0] # Corrected from [8] to [0] for values

    expected_q = rewards_tensor + GAMMA * next_q_values * (1 - dones_tensor)

    loss_fn = nn.MSELoss() # Corrected from nn.HSELoss()
    loss = loss_fn(q_values, expected_q.detach())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


print("DHANSHREE CHAUDHARI | 221A033 | 08")

# The last line 'την gym.nake("ALE/Pong-vs", render_node="rgb_array")' was causing a syntax error
# and its intent was unclear. It has been removed/commented out for now.


DHANSHREE CHAUDHARI | 221A033 | 08
DHANSHREE CHAUDHARI | 221A033 | 08
DHANSHREE CHAUDHARI | 221A033 | 08


In [ ]:
env.reset()

(array([[[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [109, 118,  43],
         [109, 118,  43],
         [109, 118,  43]],
 
        [[109, 118,  43],
         [109, 118,  43],
         [109, 118,  43],
         ...,
         [109, 118,  43],
         [109, 118,  43],
         [109, 118,  43]],
 
        [[109, 118,  43],
         [109, 118,  43],
         [109, 118,  43],
         ...,
         [109, 118,  43],
         [109, 118,  43],
         [109, 118,  43]],
 
        ...,
 
        [[ 53,  95,  24],
         [ 53,  95,  24],
         [ 53,  95,  24],
         ...,
         [ 53,  95,  24],
         [ 53,  95,  24],
         [ 53,  95,  24]],
 
        [[ 53,  95,  24],
         [ 53,  95,  24],
         [ 53,  95,  24],
         ...,
         [ 53,  95,  24],
         [ 53,  95,  24],
         [ 53,  95,  24]],
 
        [[ 53,  95,  24],
         [ 53,  95,  24],
         [ 53,  95,  24],
         ...,
         [ 53,  95,  24],
  

TRAINING LOOP


In [ ]:
# Main Training Loop

num_frames = 1000000 # Total frames for training
episode_rewards = deque(maxlen=100) # To track average rewards

state_raw, _ = env.reset()

# Preprocessing for initial state: Convert to grayscale, resize, and stack frames
def preprocess_state(raw_state):
    # Convert to grayscale
    gray_state = np.mean(raw_state, axis=2).astype(np.uint8)
    # Resize to 84x84 (Atari standard) using skimage
    resized_state = resize(gray_state, (84, 84), anti_aliasing=True).astype(np.uint8) # Resize returns float, convert back to uint8
    # For the first state, stack it 4 times
    stacked_state = np.stack([resized_state] * 4, axis=0)
    return stacked_state


state = preprocess_state(state_raw)

for frame_idx in range(1, num_frames + 1):
    epsilon = epsilon_by_frame(frame_idx)
    action = select_action(state, epsilon)

    next_state_raw, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

    next_state_processed = preprocess_state(next_state_raw)

    # Update the stacked state for the next step
    if not done:
        next_state = np.concatenate((state[1:], np.expand_dims(next_state_processed[0], axis=0)), axis=0)
    else:
        next_state = np.zeros(input_shape) # Reset to black screens or zeros

    replay_buffer.push(state, action, reward, next_state, done)

    state = next_state

    train_step()

    if frame_idx % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if done:
        state_raw, _ = env.reset()
        state = preprocess_state(state_raw)


KeyboardInterrupt: 